# 🧱 Introduction to Databricks — Project Showcase

![Databricks](https://img.shields.io/badge/Platform-Databricks-FF3621?logo=databricks&logoColor=white)
![Concept](https://img.shields.io/badge/Focus-Lakehouse%20Architecture-blueviolet)
![Status](https://img.shields.io/badge/Status-Completed-brightgreen)

**Course:** Introduction to Databricks (DataCamp)
**What this covers:** the conceptual foundations of the Databricks Data Intelligence Platform — how it evolved from the data warehouse/data lake split into a single **Lakehouse**, how it is architected (Control Plane vs Compute Plane), how data, compute, and analytics are organized inside it, and who administers what.

This notebook is my own synthesis of the course, organized chapter by chapter, written as a reference I can come back to whenever I spin up a real Databricks workspace.


## Chapter 1 — The Databricks Data Intelligence Platform

Databricks didn't start as a single unified platform — it grew out of a real pain point in data teams: warehouses were great for BI but couldn't handle unstructured data or ML workloads, while data lakes were flexible and cheap but lacked the reliability (ACID transactions, schema enforcement) that BI teams needed. The **Lakehouse** architecture is Databricks' answer to that split: one system that has the reliability and performance of a warehouse *and* the flexibility and scale of a lake.

The **Data Intelligence Platform** is the next step on top of that: same Lakehouse core, but with built-in AI and first-class support for custom AI applications layered directly onto the data.


### 1.1 — Why unify the warehouse and the lake?

| | Data Warehouse | Data Lake | Lakehouse |
|---|---|---|---|
| Data types | Structured only | Structured, semi-structured, unstructured | All of the above |
| Reliability | ACID transactions | Weak / none | ACID transactions (via Delta) |
| Cost | High | Low | Low |
| Best for | BI / reporting | ML / data science | BI **and** ML on the same copy of data |

**Key benefits I took away:**
- **Unification** — every use case, from AI to BI, runs on the same data without duplicating it into separate systems.
- **Multi-Cloud** — the platform layer sits on top of AWS, Azure, or GCP, so there's no hard lock-in to one cloud.
- **Collaborative** — every data persona (engineer, analyst, scientist) can work in the same workspace in real time, instead of handing files off between disconnected tools.
- **Open-source foundation** — everything runs on Apache Spark, with first-class support for Python, R, Scala, and SQL.


### 1.2 — Architecture: Control Plane vs Compute Plane

The architecture is split into two planes, and understanding *who owns what* here matters a lot for security and cost conversations:

- **Control Plane** — owned and hosted by Databricks in your cloud region. It hosts the UI, the notebooks, and the general orchestration logic, and it's responsible for spinning up and coordinating compute nodes.
- **Compute Plane** — owned by the customer (me / my organization). This is where the actual data lives and where processing happens, inside my own cloud account, my own networking, and my own applications.

```text
┌───────────────────────────┐        ┌──────────────────────────────┐
│        Control Plane       │        │          Compute Plane        │
│  (Databricks-managed)      │◄──────►│  (Customer-owned environment) │
│  UI · Notebooks · Orchestr.│        │  Data storage · Clusters       │
└───────────────────────────┘        └──────────────────────────────┘
```

This split is exactly what makes the **Classic vs Serverless** cluster decision meaningful later in Chapter 2 — Classic clusters run in *my* Compute Plane, Serverless clusters run in Databricks' Control Plane.


### 1.3 — Administering a workspace

The course draws a clear line between two admin roles:

- **Account Administrator** — operates at the org level, from the Account Console (`accounts.cloud.databricks.com`): creates and manages **workspaces**, governs who can access which workspace, and manages the subscription/billing.
- **Workspace Administrator** — operates inside a single workspace: manages user identities within that workspace, and creates/manages the compute resources (clusters, warehouses) available to it.

Two other admin-adjacent tools worth remembering:
- **Partner Connect** — a UI-based way to connect the workspace to partner technologies (BI tools, ingestion tools) without hand-rolling API integrations.
- **Databricks Marketplace** — lets you discover and pull in third-party datasets directly into your catalog, instead of sourcing and re-uploading them manually.

**Takeaway:** before touching any data, it's worth knowing whether I'm operating as an account admin, a workspace admin, or neither — it changes what I'm even allowed to click on.


## Chapter 2 — Data & Compute in the Data Intelligence Platform

This chapter is where the platform stops being an abstract diagram and turns into two very concrete things I'll actually touch: **how data is stored and governed**, and **what actually runs my queries**.


### 2.1 — The three kinds of data I'll run into

| Kind | Structure | Typical formats | Example |
|---|---|---|---|
| **Structured** | Fixed rows & columns | `.csv`, Parquet, Delta, database tables | A `people` table with `id, name, occupation, location` |
| **Semi-structured** | Some structure, flexible content | JSON, XML, HTML | A JSON array of `{id, name, occupation, location}` objects |
| **Unstructured** | Little to no structure | JPEG, PNG, MP4, PDF, DOC | An image or a scanned PDF |

Recognizing which bucket a source falls into is the first decision point for *how* I'll land it — structured and semi-structured data usually go straight into Delta tables, while unstructured data is more often stored as files and referenced by path (or processed with a specialized pipeline, e.g. OCR before it becomes structured).


### 2.2 — Delta as the storage foundation

**Delta** (`delta.io`) is the open-source storage format underneath almost everything in the Lakehouse:
- It's a collection of **Parquet** files plus a **JSON transaction log** that tracks every change.
- That log is what makes Delta **fully ACID-compliant** — reads never see a half-written update.
- It supports **both batch and streaming** on the same table, which is what lets a single Delta table act as source of truth for a scheduled job *and* a live dashboard.

### 2.3 — Unity Catalog for governance

**Unity Catalog** is the governance layer on top of that storage: one place to manage permissions across every data asset in the platform, using familiar SQL-style statements:

```sql
GRANT SELECT ON TABLE main.sales.orders TO `analytics_team`;
REVOKE SELECT ON TABLE main.sales.orders FROM `contractor_group`;
SHOW GRANTS ON TABLE main.sales.orders;
```

The **Catalog Explorer** is the UI on top of Unity Catalog — a single place to browse tables, manage those permissions visually, and inspect **data lineage** (which jobs/notebooks read from or wrote to a given table).


### 2.4 — Compute: Apache Spark, clusters, and runtime

**Apache Spark** is the open-source distributed computing engine Databricks was built around (created by Databricks' own co-founders). It exposes APIs in Python, SQL, Scala, and R, and scales from a laptop-sized job to a cluster processing terabytes.

**Cluster types — Classic vs Serverless:**

| | Classic | Serverless |
|---|---|---|
| Where compute runs | Compute Plane (my cloud account) | Control Plane (Databricks-managed) |
| Startup time | Slower | Fast |
| Pros | Compute & security stay inside my environment; can reuse existing compute pools | Fastest performance, always the latest features, Databricks tunes it over time |
| Cons | Slower cold start | Compute isn't physically inside my own environment |

**Single-node vs multi-node:**
- **Single-node** — just a driver, no workers. Can still run Spark, but also runs single-node frameworks like pandas directly. Best for smaller datasets.
- **Multi-node** — a driver plus one or more worker nodes, so Spark can actually distribute the work. Best for larger datasets.

**Databricks Runtime** is installed on every cluster — it's an optimized build of Spark (with **Photon**, Databricks' native vectorized query engine, for faster SQL) bundled with common libraries (pandas, dplyr, scikit-learn) and the plumbing needed to talk to the rest of the platform. General rule of thumb from the course: default to the most recent **LTS (Long Term Support)** runtime version unless there's a specific reason not to.


## Chapter 3 — Analytics in the Data Intelligence Platform

The final chapter ties data + compute together into the actual day-to-day analytics workflow — the languages I can use, the tools I write code in, and how SQL specifically fits into the platform.


### 3.1 — Supported languages

| Language | Typically used for |
|---|---|
| **Scala** | Data engineering (Spark's native language, built on the JVM) |
| **Python** | All use cases — engineering, analysis, ML |
| **SQL** | Data engineering and BI/analytics |
| **R** | Data science use cases |

### 3.2 — Where the code actually gets written

- **Databricks Notebooks** — an enhanced, Databricks-native evolution of Jupyter notebooks; the default place to mix Python/SQL/Scala/R with markdown documentation and visualizations in one document.
- **SQL Editor** — a dedicated, spreadsheet-like environment purpose-built for writing and running SQL against Unity Catalog tables, separate from the notebook interface.
- **Databricks Connect** — lets me "bring my own IDE": write code locally in my editor of choice while the actual execution happens against a real Databricks cluster.


### 3.3 — Databricks SQL: the platform's data warehouse layer

**Databricks SQL** is what makes the Lakehouse usable by pure SQL/BI users without touching a notebook at all:

```sql
-- Reading raw files directly as a table
SELECT *
FROM json.`/Volumes/catalog_name/schema_name/volume_name/path/data`;

-- Querying a governed Unity Catalog table
SELECT *
FROM catalog_name.schema_name.table_name;

-- Materializing a new governed table from a query
CREATE TABLE catalog_name.schema_name.table_name AS
SELECT *
FROM catalog_name.schema_name.source_table
WHERE ...;
```

**Why it matters as a warehousing layer:**
- Fully integrated with the same governed data everyone else on the platform uses — no separate copy.
- Scalable, performant compute purpose-built for SQL (powered by **Photon**).
- A UI genuinely built for analysts, with built-in visualizations, not just a raw query console.
- Connects directly to external BI tools (Power BI, Tableau, etc.) through Partner Connect, so the governed Lakehouse data can feed dashboards outside Databricks too.


## ✅ Skills demonstrated in this module

- Explaining the Lakehouse architecture and why it replaces the warehouse/lake split, in terms of unification, multi-cloud portability, collaboration, and open-source foundations.
- Distinguishing the **Control Plane** from the **Compute Plane**, and mapping that distinction onto real decisions (e.g. Classic vs Serverless clusters).
- Identifying **Account Administrator** vs **Workspace Administrator** responsibilities inside a Databricks organization.
- Classifying data as structured, semi-structured, or unstructured, and connecting that to storage format choices.
- Explaining how **Delta** provides ACID transactions on top of Parquet, and how **Unity Catalog** governs access and lineage over that data.
- Choosing between cluster types (Classic/Serverless, single-node/multi-node) based on workload and dataset size, and knowing what the Databricks Runtime + Photon add on top of raw Spark.
- Mapping the four supported languages (Python, SQL, Scala, R) to the workflows they're each best suited for.
- Explaining how **Databricks SQL** functions as a governed, BI-ready data warehouse layer on top of the Lakehouse.
